# Part C - Django Framework Overview

> ℹ️ **Note:** This section is conceptual. The code examples are **read-only** -  
> running them requires a full Django project directory setup.

## Table of Contents

**Part C : Framework Overview**

15. [Django Framework Overview](#15-django-framework-overview)

**Wrap-up**

16. [Common Mistakes Reference](#16-common-mistakes-reference)
17. [Best Practices Summary](#17-best-practices-summary)
18. [Key Takeaways](#18-key-takeaways)
19. [Unit Summary](#19-unit-summary)


<a id='s22'></a>
## Section 22: What Is Django?

> *"The web framework for perfectionists with deadlines."*

**Django** is a high-level Python web framework that handles:  
- URL routing  
- Database access via ORM (no SQL needed)  
- HTML rendering via templates  
- Authentication, admin interface, forms, caching, and more

**Used by:** Instagram, Pinterest, Disqus, Mozilla, Spotify (backend tools)

**How it connects to this course:**

| Previous Unit | Django Component |
|---|---|
| Unit 2 - OOP (classes) | Models - Python classes that map to DB tables |
| Unit 6 - SQLite / Databases | ORM - query the DB in pure Python |
| Unit 7 - Networking / HTTP | Views and URL routing - handle web requests |
| Unit 8 - Threads / Concurrency | Django handles concurrency automatically |

Note: 
> 🌐 **Django is the destination.**  
> Every unit in this course has built one piece of what a web framework needs.

<a id='s23'></a>
## Section 23: MVT Architecture

Django uses the **Model–View–Template (MVT)** pattern:

```
Browser Request
      │
      ▼
  URL    ────▶        View           ────▶   Template 
Routing               (Logic)                  (HTML)     
                         │
                  ┌──────▼───────┐
                  │    Model     │
                  │  (Database)  │
                  └──────────────┘
```

| Component | File | Responsibility |
|---|---|---|
| **Model** | `models.py` | Define data structure; interact with database |
| **View** | `views.py` | Handle HTTP requests; apply business logic |
| **Template** | `templates/*.html` | Render the final webpage |
| **URL conf** | `urls.py` | Map URL paths to view functions |

**Request flow example:**
1. Browser sends `GET /search/?q=Python`
2. `urls.py` routes it to `book_search(request)`
3. View queries: `Book.objects.filter(title__icontains='Python')`
4. View passes results to template
5. Template renders HTML → browser displays the page

<a id='s24'></a>
## Section 24: Django Models and ORM

Each **Django Model** is a Python class that automatically becomes a database table.  
The ORM (Object-Relational Mapper) generates all the SQL for you.

```python
# models.py
from django.db import models

class Student(models.Model):
    name     = models.CharField(max_length=100)
    email    = models.EmailField(unique=True)
    enrolled = models.DateField(auto_now_add=True)

class Book(models.Model):
    title  = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    copies = models.IntegerField(default=1)

class BorrowRecord(models.Model):
    student  = models.ForeignKey(Student, on_delete=models.CASCADE)
    book     = models.ForeignKey(Book,    on_delete=models.CASCADE)
    due_back = models.DateField()
    returned = models.BooleanField(default=False)
```

**ORM queries - pure Python, no SQL:**

```python
# Create
alice = Student.objects.create(name='Alice', email='alice@uni.edu')

# Read all
all_students = Student.objects.all()

# Filter
overdue = BorrowRecord.objects.filter(due_back__lt=today, returned=False)

# Update
alice.name = 'Alice Smith'
alice.save()

# Delete
alice.delete()
```

> **Connection to Unit 6:** The `ForeignKey` relationships are the same  
> relational concepts from your SQLite work - now expressed as Python class attributes.

<a id='s25'></a>
## Section 25: Django Views and Templates

**View - handles the HTTP request and prepares data:**

```python
# views.py
from django.shortcuts import render
from .models import Book

def book_search(request):
    query   = request.GET.get('q', '')   # get URL parameter
    results = []
    if query:
        results = Book.objects.filter(title__icontains=query)
    return render(request, 'library/search.html',
                  {'query': query, 'results': results})
```

**Template - renders the HTML:**

```html
<!-- templates/library/search.html -->
<h1>Library Search</h1>
<form method="get">
  <input name="q" value="{{ query }}">
  <button>Search</button>
</form>

{% if results %}
  <p>Found {{ results|length }} book(s):</p>
  <ul>
  {% for book in results %}
    <li>{{ book.title }} - {{ book.author }}</li>
  {% endfor %}
  </ul>
{% elif query %}
  <p>No results for "{{ query }}".</p>
{% endif %}
```

**URL routing:**

```python
# urls.py
from django.urls import path
from . import views

urlpatterns = [
    path('search/', views.book_search, name='search'),
]
```

**Template syntax:**

| Syntax | Purpose |
|---|---|
| `{{ variable }}` | Output a value |
| `{% if condition %}` | Conditional block |
| `{% for item in list %}` | Loop |
| `{{ value\|filter }}` | Apply a filter (e.g., `\|length`, `\|upper`) |

---
# Practice Tasks

<a id='practice'></a>
## Section 26: Practice Tasks

Complete each task in the code cell below it.

### Task 1: Regex - Phone Number Extractor
Write a function that extracts all UK mobile phone numbers from a string.  
UK mobiles start with `07` followed by 9 more digits (sometimes with a space after the first 4 digits).

Example: `07700 900123` or `07911654321`

In [ ]:
import re

def extract_uk_phones(text):
    """
    Extract UK mobile numbers starting with 07 (10 digits total,
    optional space after first 5 digits).
    Returns a list of phone numbers as strings.
    """
    # TODO: write your pattern and use re.findall()
    pattern = r""  # replace with your pattern
    return re.findall(pattern, text)

# Test your function
test = (
    "Call Alice on 07700 900123 or Bob on 07911654321. "
    "Not a phone: 12345 or 0800 123 456."
)
result = extract_uk_phones(test)
print("Found:", result)
# Expected: ['07700 900123', '07911654321'] (or similar valid matches)

# Hint: \b07\d{3}\s?\d{6}\b

### Task 2: Regex - Student Record Parser
Each line has the format: `NAME: SCORE / GRADE`  
Example: `Alice Smith: 78 / B`

Write a function that parses a multi-line string and returns a list of dicts:  
`[{'name': 'Alice Smith', 'score': '78', 'grade': 'B'}, ...]`

In [ ]:
import re

def parse_student_records(text):
    """
    Parse lines of the format 'NAME: SCORE / GRADE'.
    Returns a list of dicts with keys: name, score, grade.
    """
    records = []
    # TODO: Use re.finditer() or re.findall() with named groups
    # Hint: r"(?P<name>[\w ]+):\s*(?P<score>\d+)\s*/\s*(?P<grade>[A-F])"
    return records

data = """
Alice Smith: 78 / B
Bob Jones: 91 / A
Charlie Brown: 55 / D
Diana Prince: 88 / B
"""

results = parse_student_records(data)
for r in results:
    print(r)
# Expected:
# {'name': 'Alice Smith', 'score': '78', 'grade': 'B'}
# {'name': 'Bob Jones', 'score': '91', 'grade': 'A'}
# ...

### Task 3: Threading - Parallel Library Search
The library has 4 databases to search. Each search takes a different amount of time.  
Use threads to search all 4 databases concurrently and collect the results.

In [ ]:
import threading
import time
import random

# Simulate searching a database (takes 0.5–2.0 seconds)
def search_database(db_name, query, results_list, lock):
    duration = random.uniform(0.5, 2.0)
    time.sleep(duration)     # simulate DB search time
    # TODO: Add a fake result to results_list safely (use the lock)
    result = f"{db_name}: Found '{query}' in {duration:.1f}s"
    # Add result here - remember to use the lock!
    pass

def parallel_search(query):
    databases  = ["Local DB", "National DB", "Archive DB", "Digital DB"]
    results    = []
    lock       = threading.Lock()
    threads    = []
    start      = time.perf_counter()

    # TODO: Create and start one thread per database

    # TODO: Join all threads

    elapsed = time.perf_counter() - start
    return results, elapsed

found, seconds = parallel_search("Python Programming")
print(f"Search completed in {seconds:.1f}s")
for r in found:
    print(f"  {r}")

---
<a id='debug'></a>
## Section 27: Debugging Exercises

Each code block below contains a **deliberate bug**.  
Find the bug, explain why it is wrong, and fix it.

### Debug 1: Regex - Wrong Result

In [ ]:
import re

# Bug: the sub() result is being discarded
records = "Meeting on 15/03/2024 and 22/02/2024."

re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\2-\1", records)  # BUG!

print("Records:", records)   # still shows original format

# TODO: Fix the bug
# Hint: sub() returns a new string - it does NOT modify 'records' in place

### Debug 2: Regex - Greedy Pattern

In [ ]:
import re

# Bug: greedy .* captures too much
html = "<b>Name</b> and <b>Score</b>"
bold_text = re.findall(r"<b>.*</b>", html)
print("Found:", bold_text)
# Got: ['<b>Name</b> and <b>Score</b>']  -- too greedy!
# Expected: ['<b>Name</b>', '<b>Score</b>']

# TODO: Fix the pattern to use non-greedy matching

### Debug 3: Threads - Missing Join

In [ ]:
import threading
import time

results = []

def fetch_data(item):
    time.sleep(0.5)   # simulate network delay
    results.append(f"Data: {item}")

# Bug: threads are started but never joined
threads = []
for i in range(5):
    t = threading.Thread(target=fetch_data, args=(i,))
    threads.append(t)
    t.start()    # start all threads

# BUG: join is missing!
print("Results:", results)  # often empty - threads haven't finished!

# TODO: Add the missing join loop before the print statement

### Debug 4: Threads - Race Condition

In [ ]:
import threading

# Bug: shared counter modified without a lock
counter = 0

def increment(times):
    global counter
    for _ in range(times):
        counter += 1   # BUG: not thread-safe!

threads = [threading.Thread(target=increment, args=(50_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Expected: 200000")
print(f"Got:      {counter}")

# TODO: Fix by adding a Lock

---
<a id='challenges'></a>
## Section 28: Mini Coding Challenges

### Challenge 1: Password Strength Checker

Write a `check_password(password)` function that returns a list of **failed requirements**.  
The password must:

- Be at least 8 characters long
- Contain at least one uppercase letter `[A-Z]`
- Contain at least one digit `\d`
- Contain at least one special character from `!@#$%^&*`

Use `re.search()` for each requirement.

In [ ]:
import re

def check_password(password):
    """
    Check password strength.
    Returns: list of strings describing each failed requirement.
             Empty list means the password is strong.
    """
    failures = []
    # TODO: Check each requirement using re.search()
    # 1. At least 8 characters
    # 2. At least one uppercase letter
    # 3. At least one digit
    # 4. At least one special character from !@#$%^&*
    return failures

# Test
passwords = [
    "abc",
    "password123",
    "Password1",
    "Password1!",
    "Str0ng!Pass",
]

for p in passwords:
    issues = check_password(p)
    if issues:
        print(f"WEAK   '{p}': {issues}")
    else:
        print(f"STRONG '{p}'")

### Challenge 2: Threaded Word Frequency Counter

Given a list of long strings (e.g., book chapters), use threads to count  
word frequencies in each string **concurrently**, then merge the results.

Use a `Lock` to protect the shared frequency dictionary.

In [ ]:
import threading
import re
from collections import Counter

def count_words_in_chapter(chapter_text, shared_counts, lock):
    """
    Count words in chapter_text and add to shared_counts.
    Use the lock to protect shared_counts.
    """
    # Extract words (lowercase), ignore short words
    words = re.findall(r"\b[a-zA-Z]{4,}\b", chapter_text.lower())
    local_counts = Counter(words)

    # TODO: Acquire lock and merge local_counts into shared_counts
    pass

# Sample 'chapters'
chapters = [
    "Alice studied Python programming every evening after college. Python is amazing.",
    "Bob practiced threading and regex in Python. Threading helps performance greatly.",
    "Charlie loved Python data science and studied regex patterns daily for programming.",
]

shared_word_counts = Counter()
counter_lock       = threading.Lock()
threads = []

# TODO: Create one thread per chapter and start them all

# TODO: Join all threads

print("Top 10 words across all chapters:")
for word, count in shared_word_counts.most_common(10):
    print(f"  {word:15s} {count}")

### Challenge 3: Log File Analyser

Given a multi-line log string, use regex to extract and summarise:

1. All ERROR lines
2. All timestamps (format: `HH:MM:SS`)
3. The IP addresses that appear in WARNING lines
4. Total count of each log level (INFO, WARNING, ERROR)

Use `re.findall()` and `re.search()` appropriately.

In [ ]:
import re
from collections import Counter

log = """
10:15:02 INFO  User alice logged in from 192.168.1.1
10:15:45 WARNING  Failed login attempt from 10.0.0.5
10:16:03 ERROR  Database connection timeout
10:17:11 INFO  User bob logged in from 192.168.1.2
10:18:30 WARNING  Rate limit exceeded from 10.0.0.5
10:19:01 ERROR  File not found: /data/records.db
10:19:45 INFO  Backup completed successfully
10:20:12 WARNING  Disk usage at 85% on 10.0.0.3
"""

# TODO 1: Extract all ERROR lines
error_lines = []  # use re.findall()

# TODO 2: Extract all timestamps (HH:MM:SS)
timestamps = []   # use re.findall()

# TODO 3: Extract IP addresses from WARNING lines only
warning_ips = []  # hint: find WARNING lines first, then search for IPs

# TODO 4: Count log levels
level_counts = Counter()  # count INFO, WARNING, ERROR

print("Error lines:", error_lines)
print("Timestamps:",  timestamps)
print("Warning IPs:", warning_ips)
print("Level counts:", dict(level_counts))

---
<a id='summary'></a>
## Section 29: Unit Summary and Checklist

### Part A - Regular Expressions

| Topic | Key Points |
|---|---|
| Raw strings | Always `r"..."` - never omit the `r` prefix |
| `re.split()` | Split on a pattern; handles multiple separators |
| Character classes | `[abc]` = any of a, b, c; `[^abc]` = NOT a, b, c |
| Predefined | `\d` digit, `\w` word char, `\s` whitespace, `\b` boundary |
| Quantifiers | `*` 0+, `+` 1+, `?` 0 or 1, `{n}` exact, `{n,m}` range |
| Non-greedy | Add `?` after quantifier: `.*?` matches as little as possible |
| `re.match()` | Only checks start; use `^...$` for whole-string validation |
| `re.search()` | Finds pattern anywhere; returns first match |
| `re.findall()` | Returns list of all matches (or tuples with groups) |
| `re.sub()` | Returns new string - always assign the result |
| `re.compile()` | Compile once for reuse; supports flags (re.I, re.M, re.S) |

### Part B - Threads

| Topic | Key Points |
|---|---|
| Process vs Thread | Threads share memory; faster communication but race conditions |
| GIL | Threads = I/O-bound; multiprocessing = CPU-bound |
| Three-step pattern | Create → Start → Join |
| Multiple threads | Start ALL then Join ALL - never start-join in same loop |
| Daemon | `daemon=True` = auto-killed when main thread ends |
| Race condition | Unprotected shared writes produce non-deterministic results |
| Lock | `with lock:` protects critical section; only one thread at a time |
| Semaphore | `Semaphore(n)` - limits N concurrent threads |
| Event | `event.set()` signals all threads checking `is_set()` |
| Exceptions | Unhandled exceptions in threads are SILENT - use try/except |

### Part C - Django

| Topic | Key Points |
|---|---|
| MVT | Model (data), View (logic), Template (HTML) |
| Model | Python class → DB table; fields = columns |
| ORM | Pure Python queries - no SQL needed for standard operations |
| View | Receives HTTP request, queries DB, returns rendered template |
| Template | HTML with `{{ }}` variables and `{% %}` logic tags |

---

### Self-Assessment Checklist

Tick each item when you feel confident:

- [ ] I can write a regex pattern using `\d`, `\w`, `\s`, `\b`, `[...]`, `{n,m}`
- [ ] I understand when to use `match()` vs `search()` vs `findall()`
- [ ] I always use raw strings `r"..."` for patterns
- [ ] I always assign `re.sub()` result: `text = re.sub(..., text)`
- [ ] I can create a thread, start it, and join it
- [ ] I understand the GIL and know when threads help vs hurt
- [ ] I can protect shared data with `with lock:`
- [ ] I can use `Semaphore` to limit concurrent access
- [ ] I can use `Event` to signal between threads
- [ ] I can explain Django's MVT architecture

---

**Next Steps:**

- 📘 **Lab:** Regex email validator + threaded library checkout simulation
- 📝 **Assignment:** Multi-threaded data processor with regex parsing
- 🌐 **Self-study:** [Django Official Tutorial](https://docs.djangoproject.com/en/stable/intro/tutorial01/)
- 🔗 **Regex practice:** [regex101.com](https://regex101.com)

*BT151CO - Object-Oriented Programming with Python*

<a id='s23'></a>
## Section 23: MVT Architecture

Django uses the **Model–View–Template (MVT)** pattern:

```
Browser Request
      │
      ▼
┌──────────┐     ┌──────────────┐     ┌──────────────┐
│   URL    │────▶│    View      │────▶│   Template   │
│ Routing  │     │  (Logic)     │     │   (HTML)     │
└──────────┘     └──────┬───────┘     └──────────────┘
                        │
                 ┌──────▼───────┐
                 │    Model     │
                 │  (Database)  │
                 └──────────────┘
```

| Component | File | Responsibility |
|---|---|---|
| **Model** | `models.py` | Define data structure; interact with database |
| **View** | `views.py` | Handle HTTP requests; apply business logic |
| **Template** | `templates/*.html` | Render the final webpage |
| **URL conf** | `urls.py` | Map URL paths to view functions |

**Request flow example:**
1. Browser sends `GET /search/?q=Python`
2. `urls.py` routes it to `book_search(request)`
3. View queries: `Book.objects.filter(title__icontains='Python')`
4. View passes results to template
5. Template renders HTML → browser displays the page

<a id='s24'></a>
## Section 24: Django Models and ORM

Each **Django Model** is a Python class that automatically becomes a database table.  
The ORM (Object-Relational Mapper) generates all the SQL for you.

```python
# models.py
from django.db import models

class Student(models.Model):
    name     = models.CharField(max_length=100)
    email    = models.EmailField(unique=True)
    enrolled = models.DateField(auto_now_add=True)

class Book(models.Model):
    title  = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    copies = models.IntegerField(default=1)

class BorrowRecord(models.Model):
    student  = models.ForeignKey(Student, on_delete=models.CASCADE)
    book     = models.ForeignKey(Book,    on_delete=models.CASCADE)
    due_back = models.DateField()
    returned = models.BooleanField(default=False)
```

**ORM queries - pure Python, no SQL:**

```python
# Create
alice = Student.objects.create(name='Alice', email='alice@uni.edu')

# Read all
all_students = Student.objects.all()

# Filter
overdue = BorrowRecord.objects.filter(due_back__lt=today, returned=False)

# Update
alice.name = 'Alice Smith'
alice.save()

# Delete
alice.delete()
```

> **Connection to Unit 6:** The `ForeignKey` relationships are the same  
> relational concepts from your SQLite work - now expressed as Python class attributes.

<a id='s25'></a>
## Section 25: Django Views and Templates

**View - handles the HTTP request and prepares data:**

```python
# views.py
from django.shortcuts import render
from .models import Book

def book_search(request):
    query   = request.GET.get('q', '')   # get URL parameter
    results = []
    if query:
        results = Book.objects.filter(title__icontains=query)
    return render(request, 'library/search.html',
                  {'query': query, 'results': results})
```

**Template - renders the HTML:**

```html
<!-- templates/library/search.html -->
<h1>Library Search</h1>
<form method="get">
  <input name="q" value="{{ query }}">
  <button>Search</button>
</form>

{% if results %}
  <p>Found {{ results|length }} book(s):</p>
  <ul>
  {% for book in results %}
    <li>{{ book.title }} - {{ book.author }}</li>
  {% endfor %}
  </ul>
{% elif query %}
  <p>No results for "{{ query }}".</p>
{% endif %}
```

**URL routing:**

```python
# urls.py
from django.urls import path
from . import views

urlpatterns = [
    path('search/', views.book_search, name='search'),
]
```

**Template syntax:**

| Syntax | Purpose |
|---|---|
| `{{ variable }}` | Output a value |
| `{% if condition %}` | Conditional block |
| `{% for item in list %}` | Loop |
| `{{ value\|filter }}` | Apply a filter (e.g., `\|length`, `\|upper`) |

---
# Practice Tasks

<a id='practice'></a>
## Section 26: Practice Tasks

Complete each task in the code cell below it.

### Task 1: Regex - Phone Number Extractor
Write a function that extracts all UK mobile phone numbers from a string.  
UK mobiles start with `07` followed by 9 more digits (sometimes with a space after the first 4 digits).

Example: `07700 900123` or `07911654321`

In [ ]:
import re

def extract_uk_phones(text):
    """
    Extract UK mobile numbers starting with 07 (10 digits total,
    optional space after first 5 digits).
    Returns a list of phone numbers as strings.
    """
    # TODO: write your pattern and use re.findall()
    pattern = r""  # replace with your pattern
    return re.findall(pattern, text)

# Test your function
test = (
    "Call Alice on 07700 900123 or Bob on 07911654321. "
    "Not a phone: 12345 or 0800 123 456."
)
result = extract_uk_phones(test)
print("Found:", result)
# Expected: ['07700 900123', '07911654321'] (or similar valid matches)

# Hint: \b07\d{3}\s?\d{6}\b

### Task 2: Regex - Student Record Parser
Each line has the format: `NAME: SCORE / GRADE`  
Example: `Alice Smith: 78 / B`

Write a function that parses a multi-line string and returns a list of dicts:  
`[{'name': 'Alice Smith', 'score': '78', 'grade': 'B'}, ...]`

In [ ]:
import re

def parse_student_records(text):
    """
    Parse lines of the format 'NAME: SCORE / GRADE'.
    Returns a list of dicts with keys: name, score, grade.
    """
    records = []
    # TODO: Use re.finditer() or re.findall() with named groups
    # Hint: r"(?P<name>[\w ]+):\s*(?P<score>\d+)\s*/\s*(?P<grade>[A-F])"
    return records

data = """
Alice Smith: 78 / B
Bob Jones: 91 / A
Charlie Brown: 55 / D
Diana Prince: 88 / B
"""

results = parse_student_records(data)
for r in results:
    print(r)
# Expected:
# {'name': 'Alice Smith', 'score': '78', 'grade': 'B'}
# {'name': 'Bob Jones', 'score': '91', 'grade': 'A'}
# ...

### Task 3: Threading - Parallel Library Search
The library has 4 databases to search. Each search takes a different amount of time.  
Use threads to search all 4 databases concurrently and collect the results.

In [ ]:
import threading
import time
import random

# Simulate searching a database (takes 0.5–2.0 seconds)
def search_database(db_name, query, results_list, lock):
    duration = random.uniform(0.5, 2.0)
    time.sleep(duration)     # simulate DB search time
    # TODO: Add a fake result to results_list safely (use the lock)
    result = f"{db_name}: Found '{query}' in {duration:.1f}s"
    # Add result here - remember to use the lock!
    pass

def parallel_search(query):
    databases  = ["Local DB", "National DB", "Archive DB", "Digital DB"]
    results    = []
    lock       = threading.Lock()
    threads    = []
    start      = time.perf_counter()

    # TODO: Create and start one thread per database

    # TODO: Join all threads

    elapsed = time.perf_counter() - start
    return results, elapsed

found, seconds = parallel_search("Python Programming")
print(f"Search completed in {seconds:.1f}s")
for r in found:
    print(f"  {r}")

---
<a id='debug'></a>
## Section 27: Debugging Exercises

Each code block below contains a **deliberate bug**.  
Find the bug, explain why it is wrong, and fix it.

### Debug 1: Regex - Wrong Result

In [ ]:
import re

# Bug: the sub() result is being discarded
records = "Meeting on 15/03/2024 and 22/02/2024."

re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\2-\1", records)  # BUG!

print("Records:", records)   # still shows original format

# TODO: Fix the bug
# Hint: sub() returns a new string - it does NOT modify 'records' in place

### Debug 2: Regex - Greedy Pattern

In [ ]:
import re

# Bug: greedy .* captures too much
html = "<b>Name</b> and <b>Score</b>"
bold_text = re.findall(r"<b>.*</b>", html)
print("Found:", bold_text)
# Got: ['<b>Name</b> and <b>Score</b>']  -- too greedy!
# Expected: ['<b>Name</b>', '<b>Score</b>']

# TODO: Fix the pattern to use non-greedy matching

### Debug 3: Threads - Missing Join

In [ ]:
import threading
import time

results = []

def fetch_data(item):
    time.sleep(0.5)   # simulate network delay
    results.append(f"Data: {item}")

# Bug: threads are started but never joined
threads = []
for i in range(5):
    t = threading.Thread(target=fetch_data, args=(i,))
    threads.append(t)
    t.start()    # start all threads

# BUG: join is missing!
print("Results:", results)  # often empty - threads haven't finished!

# TODO: Add the missing join loop before the print statement

### Debug 4: Threads - Race Condition

In [ ]:
import threading

# Bug: shared counter modified without a lock
counter = 0

def increment(times):
    global counter
    for _ in range(times):
        counter += 1   # BUG: not thread-safe!

threads = [threading.Thread(target=increment, args=(50_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Expected: 200000")
print(f"Got:      {counter}")

# TODO: Fix by adding a Lock

---
<a id='challenges'></a>
## Section 28: Mini Coding Challenges

### Challenge 1: Password Strength Checker

Write a `check_password(password)` function that returns a list of **failed requirements**.  
The password must:

- Be at least 8 characters long
- Contain at least one uppercase letter `[A-Z]`
- Contain at least one digit `\d`
- Contain at least one special character from `!@#$%^&*`

Use `re.search()` for each requirement.

In [ ]:
import re

def check_password(password):
    """
    Check password strength.
    Returns: list of strings describing each failed requirement.
             Empty list means the password is strong.
    """
    failures = []
    # TODO: Check each requirement using re.search()
    # 1. At least 8 characters
    # 2. At least one uppercase letter
    # 3. At least one digit
    # 4. At least one special character from !@#$%^&*
    return failures

# Test
passwords = [
    "abc",
    "password123",
    "Password1",
    "Password1!",
    "Str0ng!Pass",
]

for p in passwords:
    issues = check_password(p)
    if issues:
        print(f"WEAK   '{p}': {issues}")
    else:
        print(f"STRONG '{p}'")

### Challenge 2: Threaded Word Frequency Counter

Given a list of long strings (e.g., book chapters), use threads to count  
word frequencies in each string **concurrently**, then merge the results.

Use a `Lock` to protect the shared frequency dictionary.

In [ ]:
import threading
import re
from collections import Counter

def count_words_in_chapter(chapter_text, shared_counts, lock):
    """
    Count words in chapter_text and add to shared_counts.
    Use the lock to protect shared_counts.
    """
    # Extract words (lowercase), ignore short words
    words = re.findall(r"\b[a-zA-Z]{4,}\b", chapter_text.lower())
    local_counts = Counter(words)

    # TODO: Acquire lock and merge local_counts into shared_counts
    pass

# Sample 'chapters'
chapters = [
    "Alice studied Python programming every evening after college. Python is amazing.",
    "Bob practiced threading and regex in Python. Threading helps performance greatly.",
    "Charlie loved Python data science and studied regex patterns daily for programming.",
]

shared_word_counts = Counter()
counter_lock       = threading.Lock()
threads = []

# TODO: Create one thread per chapter and start them all

# TODO: Join all threads

print("Top 10 words across all chapters:")
for word, count in shared_word_counts.most_common(10):
    print(f"  {word:15s} {count}")

### Challenge 3: Log File Analyser

Given a multi-line log string, use regex to extract and summarise:

1. All ERROR lines
2. All timestamps (format: `HH:MM:SS`)
3. The IP addresses that appear in WARNING lines
4. Total count of each log level (INFO, WARNING, ERROR)

Use `re.findall()` and `re.search()` appropriately.

In [ ]:
import re
from collections import Counter

log = """
10:15:02 INFO  User alice logged in from 192.168.1.1
10:15:45 WARNING  Failed login attempt from 10.0.0.5
10:16:03 ERROR  Database connection timeout
10:17:11 INFO  User bob logged in from 192.168.1.2
10:18:30 WARNING  Rate limit exceeded from 10.0.0.5
10:19:01 ERROR  File not found: /data/records.db
10:19:45 INFO  Backup completed successfully
10:20:12 WARNING  Disk usage at 85% on 10.0.0.3
"""

# TODO 1: Extract all ERROR lines
error_lines = []  # use re.findall()

# TODO 2: Extract all timestamps (HH:MM:SS)
timestamps = []   # use re.findall()

# TODO 3: Extract IP addresses from WARNING lines only
warning_ips = []  # hint: find WARNING lines first, then search for IPs

# TODO 4: Count log levels
level_counts = Counter()  # count INFO, WARNING, ERROR

print("Error lines:", error_lines)
print("Timestamps:",  timestamps)
print("Warning IPs:", warning_ips)
print("Level counts:", dict(level_counts))

---
<a id='summary'></a>
## Section 29: Unit Summary and Checklist

### Part A - Regular Expressions

| Topic | Key Points |
|---|---|
| Raw strings | Always `r"..."` - never omit the `r` prefix |
| `re.split()` | Split on a pattern; handles multiple separators |
| Character classes | `[abc]` = any of a, b, c; `[^abc]` = NOT a, b, c |
| Predefined | `\d` digit, `\w` word char, `\s` whitespace, `\b` boundary |
| Quantifiers | `*` 0+, `+` 1+, `?` 0 or 1, `{n}` exact, `{n,m}` range |
| Non-greedy | Add `?` after quantifier: `.*?` matches as little as possible |
| `re.match()` | Only checks start; use `^...$` for whole-string validation |
| `re.search()` | Finds pattern anywhere; returns first match |
| `re.findall()` | Returns list of all matches (or tuples with groups) |
| `re.sub()` | Returns new string - always assign the result |
| `re.compile()` | Compile once for reuse; supports flags (re.I, re.M, re.S) |

### Part B - Threads

| Topic | Key Points |
|---|---|
| Process vs Thread | Threads share memory; faster communication but race conditions |
| GIL | Threads = I/O-bound; multiprocessing = CPU-bound |
| Three-step pattern | Create → Start → Join |
| Multiple threads | Start ALL then Join ALL - never start-join in same loop |
| Daemon | `daemon=True` = auto-killed when main thread ends |
| Race condition | Unprotected shared writes produce non-deterministic results |
| Lock | `with lock:` protects critical section; only one thread at a time |
| Semaphore | `Semaphore(n)` - limits N concurrent threads |
| Event | `event.set()` signals all threads checking `is_set()` |
| Exceptions | Unhandled exceptions in threads are SILENT - use try/except |

### Part C - Django

| Topic | Key Points |
|---|---|
| MVT | Model (data), View (logic), Template (HTML) |
| Model | Python class → DB table; fields = columns |
| ORM | Pure Python queries - no SQL needed for standard operations |
| View | Receives HTTP request, queries DB, returns rendered template |
| Template | HTML with `{{ }}` variables and `{% %}` logic tags |

---

### Self-Assessment Checklist

Tick each item when you feel confident:

- [ ] I can write a regex pattern using `\d`, `\w`, `\s`, `\b`, `[...]`, `{n,m}`
- [ ] I understand when to use `match()` vs `search()` vs `findall()`
- [ ] I always use raw strings `r"..."` for patterns
- [ ] I always assign `re.sub()` result: `text = re.sub(..., text)`
- [ ] I can create a thread, start it, and join it
- [ ] I understand the GIL and know when threads help vs hurt
- [ ] I can protect shared data with `with lock:`
- [ ] I can use `Semaphore` to limit concurrent access
- [ ] I can use `Event` to signal between threads
- [ ] I can explain Django's MVT architecture

---

**Next Steps:**

- 📘 **Lab:** Regex email validator + threaded library checkout simulation
- 📝 **Assignment:** Multi-threaded data processor with regex parsing
- 🌐 **Self-study:** [Django Official Tutorial](https://docs.djangoproject.com/en/stable/intro/tutorial01/)
- 🔗 **Regex practice:** [regex101.com](https://regex101.com)

*BT151CO - Object-Oriented Programming with Python*